# pydssp smoke test

Exercise the primary API of `pydssp` (PyTorch-based DSSP secondary structure assignment).


In [ ]:
# Example 1: public API surface inspection.
import pydssp

public_names = [n for n in dir(pydssp) if not n.startswith('_')]
print(f"public names ({len(public_names)}):")
for n in sorted(public_names):
    obj = getattr(pydssp, n)
    print(f"  {n}: {type(obj).__name__}")

# pydssp exposes at minimum assign + a few helpers
assert hasattr(pydssp, 'assign'), 'pydssp.assign should be available'


In [ ]:
# Example 2: build a tiny synthetic protein backbone coords tensor and run assign().
# pydssp expects coords with shape (..., L, 4, 3) for the 4 backbone atoms (N, CA, C, O).
import torch

import pydssp

torch.manual_seed(0)
L = 12
coords = torch.randn(L, 4, 3) * 2.0

ss = pydssp.assign(coords)
print(f"coords shape: {tuple(coords.shape)}")
print(f"assign output: {ss}")
print(f"output type: {type(ss).__name__}")
# output length should match the residue count
assert len(ss) == L, f"expected length {L}, got {len(ss)}"


In [ ]:
# Example 3: batched input — pydssp supports leading batch dims.
import torch

import pydssp

torch.manual_seed(1)
B, L = 2, 8
coords = torch.randn(B, L, 4, 3) * 2.0

ss = pydssp.assign(coords)
print(f"batched coords shape: {tuple(coords.shape)}")
print(f"batched assignment shape/len: {getattr(ss, 'shape', len(ss))}")
# Should produce one assignment per batch
assert len(ss) == B, f"expected batch dim {B}, got {len(ss)}"


In [ ]:
# Example 4: one-hot output mode (out_type='onehot') — returns a tensor of shape (..., L, 3).
import torch

import pydssp

torch.manual_seed(2)
L = 10
coords = torch.randn(L, 4, 3) * 2.0

try:
    onehot = pydssp.assign(coords, out_type='onehot')
    print(f"onehot shape: {tuple(onehot.shape)}")
    print(f"onehot dtype: {onehot.dtype}")
    # Each residue should have exactly one class assigned
    assert onehot.shape[-2] == L
    assert onehot.shape[-1] == 3, f"expected 3 SS classes, got {onehot.shape[-1]}"
except TypeError:
    # older API — fall back to default
    onehot = pydssp.assign(coords)
    print(f"out_type kwarg unsupported; default output: {onehot}")


In [ ]:
# Example 5: hbond-map mode + numpy round-trip.
import numpy as np
import torch

import pydssp

torch.manual_seed(3)
L = 9
coords_np = np.random.randn(L, 4, 3).astype(np.float32) * 2.0
coords = torch.from_numpy(coords_np)

# Try the hbond-map helper if exposed; otherwise just re-run assign on a numpy roundtrip
if hasattr(pydssp, 'get_hbond_map'):
    hbmap = pydssp.get_hbond_map(coords)
    print(f"hbond map shape: {tuple(hbmap.shape)}")
    assert hbmap.shape[-1] == L and hbmap.shape[-2] == L
else:
    ss = pydssp.assign(coords)
    print(f"no get_hbond_map; assign result: {ss}")

ss = pydssp.assign(coords)
assert ss is not None
print("pydssp smoke test PASSED")
